# Test split — the Fable 5.1 sparse_knn row

---
## 1 — Host and working tree

In [19]:
from pathlib import Path

%cd /home/prnamhr/projects/Style-Aware-MT

/home/prnamhr/projects/Style-Aware-MT


In [20]:
%pip install -r requirements.txt
%pip install -q anthropic==0.109.1


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
# torch 2.12 breaks the pinned-torch ABI these three ship against; the pipeline is text-only.
%pip uninstall -q -y torchvision torchaudio torchcodec

Note: you may need to restart the kernel to use updated packages.


---
## 2 — Run parameters

In [22]:
import getpass
import hashlib
import json
import logging
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone

import yaml

from src.infer.run import resolve_out_name

SPLIT = 'test'
EVAL_FILE = Path('data/splits/test.jsonl')
CONDITION = 'sparse_knn'
SRC_CFG = 'configs/commercial_fable5_sparse_knn.yaml'
PILOT_N = 20

logging.getLogger('httpx').setLevel(logging.WARNING)

In [23]:
HASHES = json.loads(Path('data/splits/hashes.json').read_text(encoding='utf-8'))
TEST_SHA = hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest()
assert TEST_SHA == HASHES['hashes']['test.jsonl'], 'test.jsonl is not the committed split'

SEGMENTS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
TEST_SRC = [r['input'] for r in SEGMENTS]
assert len(SEGMENTS) == HASHES['counts']['final']['test'] == 1322, len(SEGMENTS)
print(f'{len(SEGMENTS)} segments  {TEST_SHA[:16]}')

1322 segments  3e24e90f55e5e530


In [24]:
CFG = yaml.safe_load(Path(SRC_CFG).read_text(encoding='utf-8'))
NAME = resolve_out_name(CONDITION, CFG)
GEN = CFG['generator']

assert GEN['provider'] == 'anthropic' and GEN['model'] == 'claude-fable-5-1', GEN
assert GEN['thinking'] is True and GEN['effort'] == 'low', 'the depth control is the row parameter'
assert GEN['temperature'] is None, 'sampling is a 400 in this tier'
assert GEN['max_tokens'] == 1024, GEN

# The selection blocks are what make this the same arm as the Qwen and gpt-5.6-sol rows.
SPARSE = yaml.safe_load(Path('configs/sparse_knn.yaml').read_text(encoding='utf-8'))
for block in ('prompt', 'retrieval', 'rarity', 'sparse'):
    assert CFG[block] == SPARSE[block], (block, CFG[block], SPARSE[block])
print(f'{NAME}: {GEN["model"]} effort={GEN["effort"]}, selection identical to sparse_knn.yaml')

fable5_sparse_knn: claude-fable-5-1 effort=low, selection identical to sparse_knn.yaml


---
## 3 — The manifest this row joins

In [25]:
MANIFEST_PATH = Path('outputs/test_manifest.json')
MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert MANIFEST['eval_file']['sha256'] == TEST_SHA, 'the manifest was written against another split'
assert len(MANIFEST['rows']) == 14, len(MANIFEST['rows'])
assert NAME not in MANIFEST['rows'], f'{NAME} is already a recorded row'

INDEX = Path(MANIFEST['index']['dir'])
for f, digest in MANIFEST['index']['sha256'].items():
    assert hashlib.sha256((INDEX / f).read_bytes()).hexdigest() == digest, f
RARITY = Path(CFG['rarity']['out'])
RARITY_SHA = hashlib.sha256(RARITY.read_bytes()).hexdigest()
prior = MANIFEST['rows']['gpt56_sparse_knn']
assert MANIFEST['leakage']['fired'] is False, MANIFEST['leakage']

print(f'index verified against the 14-row manifest, {MANIFEST["index"]["meta"]["n_passages"]} passages')
print(f'rarity {RARITY} {RARITY_SHA[:12]}')
print(f'leakage {MANIFEST["leakage"]["flagged"]}/{MANIFEST["leakage"]["n"]} = '
      f'{MANIFEST["leakage"]["rate"]:.2%}, trigger not fired')

index verified against the 14-row manifest, 10860 passages
rarity results/rarity_train.json 8fa5b0b2c0db
leakage 21/1322 = 1.59%, trigger not fired


---
## 4 — Opening the seal

In [26]:
SEAL_OPEN = True

In [27]:
assert SEAL_OPEN, 'set SEAL_OPEN = True to read the sealed split'

TEST_CFG_DIR = Path('configs/test')
DERIVED = {}
# The rate check buys the next PILOT_N segments of the full pass, not a separate pilot.
for tag, limit in (('stage', PILOT_N * 2), ('full', None)):
    cfg = json.loads(json.dumps(CFG))
    assert cfg['data']['eval_file'] == 'data/splits/val.jsonl', SRC_CFG
    cfg['data']['eval_file'] = str(EVAL_FILE)
    cfg['data']['limit'] = limit
    dst = TEST_CFG_DIR / f'{Path(SRC_CFG).stem}.{tag}.yaml'
    dst.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
    DERIVED[tag] = {'path': str(dst), 'sha256': hashlib.sha256(dst.read_bytes()).hexdigest()}
    print(f'{dst}  limit={limit}  {DERIVED[tag]["sha256"][:12]}')

assert not subprocess.run(['git', 'diff', '--quiet', '--', SRC_CFG], check=False).returncode, \
    'the committed config was modified; the seal is opened by derivation, not by editing'


configs/test/commercial_fable5_sparse_knn.stage.yaml  limit=40  915083efe6c2
configs/test/commercial_fable5_sparse_knn.full.yaml  limit=None  601193ce37a1


---
## 5 — The projection

In [28]:
TOKENIZER_SLACK = 1.15
THINKING_TOK_PER_CALL = 200

prior_usage = [json.loads(Path(f'outputs/gpt56_sparse_knn_{SPLIT}{s}usage.json')
                          .read_text(encoding='utf-8')) for s in ('_', '_pilot_')]
PRIOR_PROMPT = sum(u['prompt_tokens'] for u in prior_usage)
PRIOR_OUT = sum(u['completion_tokens'] for u in prior_usage)
assert sum(u['calls'] for u in prior_usage) == len(SEGMENTS), prior_usage

in_rate, out_rate = GEN['pricing']
n = len(SEGMENTS)
proj_in = PRIOR_PROMPT * TOKENIZER_SLACK * in_rate / 1e6
proj_out = (PRIOR_OUT * TOKENIZER_SLACK + THINKING_TOK_PER_CALL * n) * out_rate / 1e6
PROJECTED_TOTAL = proj_in + proj_out
PROJECTED_RATE = PROJECTED_TOTAL / n

print(f'gpt56_sparse_knn on {SPLIT}: {PRIOR_PROMPT:,} prompt, {PRIOR_OUT:,} visible output')
print(f'at ${in_rate:.0f}/${out_rate:.0f} per MTok, x{TOKENIZER_SLACK} tokenizer slack, '
      f'+{THINKING_TOK_PER_CALL} thinking tok/call:')
print(f'  input  ${proj_in:6.2f}')
print(f'  output ${proj_out:6.2f}')
print(f'  total  ${PROJECTED_TOTAL:6.2f} over {n} calls  (${PROJECTED_RATE:.3e}/call)')

gpt56_sparse_knn on test: 2,261,560 prompt, 43,891 visible output
at $10/$40 per MTok, x1.15 tokenizer slack, +200 thinking tok/call:
  input  $ 26.01
  output $ 12.59
  total  $ 38.60 over 1322 calls  ($2.920e-02/call)


---
## 6 — The gate

In [29]:
SPEND_OK = True
AUTHORIZED_USD = 50

In [30]:
assert SEAL_OPEN and SPEND_OK, 'set SPEND_OK = True to buy this row'
assert AUTHORIZED_USD is not None, 'set AUTHORIZED_USD to the ceiling named in docs/budget.md'
assert AUTHORIZED_USD >= PROJECTED_TOTAL, (
    f'${AUTHORIZED_USD:.2f} authorized against a ${PROJECTED_TOTAL:.2f} projection'
)
print(subprocess.run(['git', 'log', '-1', '--format=%h %ad %s', '--date=short', '--',
                      'docs/budget.md'], capture_output=True, text=True).stdout)
print('the authorization this run spends against must already be a dated line in docs/budget.md')

abbe9b4 2026-08-28 docs: log the test rater passes

the authorization this run spends against must already be a dated line in docs/budget.md


In [31]:
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

---
## 7 — Pilot

In [32]:
assert SEAL_OPEN and SPEND_OK
SIDECAR = Path(f'outputs/{NAME}_{SPLIT}_usage.json')
OUT_PATH = Path(f'outputs/{NAME}_{SPLIT}.jsonl')

# These segments are paid for; a resumed re-run of the pass overwrote their usage sidecar with
# zeros, so their tokens are unrecoverable and their cost is imputed from the stage rate below.
UNRECORDED = len([x for x in OUT_PATH.open(encoding='utf-8') if x.strip()]) if OUT_PATH.exists() else 0
assert UNRECORDED == PILOT_N, f'{UNRECORDED} rows already present, expected {PILOT_N}'

r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', CONDITION,
                    '--config', DERIVED['stage']['path']], check=False)
assert r.returncode == 0, f'{NAME} stage exited {r.returncode}'

shutil.copy(SIDECAR, SIDECAR.with_name(f'{NAME}_{SPLIT}_stage_usage.json'))
u = json.loads(SIDECAR.read_text(encoding='utf-8'))
# A zero here is an unpriced model, not a free one, and it would make the guard below inert.
assert u['cost_usd'] > 0, f'{NAME}: cost_usd is 0; the model has no pricing entry'
assert u['calls'] == PILOT_N, u
STAGE = {'calls': u['calls'], 'cost_usd': u['cost_usd'], 'rate': u['cost_usd'] / u['calls']}
think = u['completion_tokens'] / u['calls']
print(f'{NAME}: {u["calls"]} calls  ${u["cost_usd"]:.4f}  ${STAGE["rate"]:.3e}/call  '
      f'projected ${PROJECTED_RATE:.3e}  x{STAGE["rate"] / PROJECTED_RATE:.2f}')
print(f'  {u["prompt_tokens"] / u["calls"]:.0f} prompt, {think:.0f} output tok/call '
      f'(assumed {THINKING_TOK_PER_CALL} thinking + visible)')
print(f'  {UNRECORDED} earlier segments carry no usage record; cost imputed at this rate')


sparse_knn: k=8 as up to 4 rarity + cosine for 40 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 709.58it/s]


  routes: {'full': 14, 'partial': 21, 'dense': 5}, mean rare slots filled: 2.45
Output name overridden: condition 'sparse_knn' -> outputs/fable5_sparse_knn_test.jsonl
resuming fable5_sparse_knn: 20/40 already done
Generating 40 translations with claude-fable-5-1 (sparse_knn) ...
  25/40
  30/40
  35/40
  40/40
Wrote outputs/fable5_sparse_knn_test.jsonl
Usage: {'calls': 20, 'prompt_tokens': 54328, 'completion_tokens': 927, 'cost_usd': 0.5804}
fable5_sparse_knn: 20 calls  $0.5804  $2.902e-02/call  projected $2.920e-02  x0.99
  2716 prompt, 46 output tok/call (assumed 200 thinking + visible)
  20 earlier segments carry no usage record; cost imputed at this rate


In [33]:
REVISED = STAGE['rate'] * len(SEGMENTS)
print(f'at the realized rate the full pass is ${REVISED:.2f} against ${PROJECTED_TOTAL:.2f} '
      f'projected and ${AUTHORIZED_USD:.2f} authorized')

assert STAGE['rate'] <= 1.25 * PROJECTED_RATE, (
    f'${STAGE["rate"]:.3e}/call is more than 1.25x the projected ${PROJECTED_RATE:.3e}; '
    f'the full pass would cost about ${REVISED:.2f}'
)
assert REVISED <= AUTHORIZED_USD, 'the realized rate exceeds the authorization'


at the realized rate the full pass is $38.36 against $38.60 projected and $50.00 authorized


In [34]:
# Blank predictions here are refusals, not crashes: the client maps stop_reason "refusal" to
# an empty string. A stage full of them means the row cannot be bought at all.
stage_rows = [json.loads(x) for x in OUT_PATH.open(encoding='utf-8') if x.strip()]
blank = [i for i, r in enumerate(stage_rows) if not r['prediction'].strip()]
assert len(stage_rows) == PILOT_N * 2, len(stage_rows)
assert len(blank) <= 2, f'{len(blank)}/{PILOT_N * 2} stage segments are empty: {blank}'
print(f'{len(stage_rows)} rows through the stage, {len(blank)} blank')
print(stage_rows[0]['prediction'][:300])


40 rows through the stage, 0 blank
Tablets of the Divine Plan


---
## 8 — The full pass

In [35]:
assert SEAL_OPEN and SPEND_OK
t0 = time.perf_counter()
r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', CONDITION,
                    '--config', DERIVED['full']['path']], check=False)
assert r.returncode == 0, f'{NAME} exited {r.returncode}'
ELAPSED = round(time.perf_counter() - t0, 1)
print(f'{NAME}: {ELAPSED / 60:.1f} min')

sparse_knn: k=8 as up to 4 rarity + cosine for 1322 ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 742.46it/s]


  routes: {'full': 596, 'partial': 625, 'dense': 101}, mean rare slots filled: 2.77
Output name overridden: condition 'sparse_knn' -> outputs/fable5_sparse_knn_test.jsonl
resuming fable5_sparse_knn: 40/1322 already done
Generating 1322 translations with claude-fable-5-1 (sparse_knn) ...
  45/1322
  50/1322
  55/1322
  60/1322
  65/1322
  70/1322
  75/1322
  80/1322
  85/1322
  90/1322
  95/1322
  100/1322
  105/1322
  110/1322
  115/1322
  120/1322
  125/1322
  130/1322
  135/1322
  140/1322
  145/1322
  150/1322
  155/1322
  160/1322
  165/1322
  170/1322
  175/1322
  180/1322
  185/1322
  190/1322
  195/1322
  200/1322
  205/1322
  210/1322
  215/1322
  220/1322
  225/1322
  230/1322
  235/1322
  240/1322
  245/1322
  250/1322
  255/1322
  260/1322
  265/1322
  270/1322
  275/1322
  280/1322
  285/1322
  290/1322
  295/1322
  300/1322
  305/1322
  310/1322
  315/1322
  320/1322
  325/1322
  330/1322
  335/1322
  340/1322
  345/1322
  350/1322
  355/1322
  360/1322
  365/1322
  370/13

In [36]:
rows = [json.loads(x) for x in OUT_PATH.open(encoding='utf-8') if x.strip()]
assert len(rows) == len(SEGMENTS), f'{NAME}: {len(rows)} rows, expected {len(SEGMENTS)}'
assert [r['input'] for r in rows] == TEST_SRC, f'{NAME}: source order differs from test.jsonl'
assert {r['model'] for r in rows} == {GEN['model']}, {r['model'] for r in rows}
blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
errors = [i for i, r in enumerate(rows) if r.get('error')]
OUTPUT_SHA = hashlib.sha256(OUT_PATH.read_bytes()).hexdigest()

full = json.loads(SIDECAR.read_text(encoding='utf-8'))
assert UNRECORDED + STAGE['calls'] + full['calls'] == len(SEGMENTS), (UNRECORDED, STAGE, full)
IMPUTED = round(STAGE['rate'] * UNRECORDED, 4)
MEASURED = round(STAGE['cost_usd'] + full['cost_usd'], 4)
SPEND = {'stage': STAGE, 'full': {k: full[k] for k in ('calls', 'cost_usd')},
         'unrecorded': {'calls': UNRECORDED, 'imputed_usd': IMPUTED,
                        'why': 'usage sidecar overwritten by a zero-call resumed re-run'},
         'total_calls': UNRECORDED + STAGE['calls'] + full['calls'],
         'measured_usd': MEASURED, 'total_usd': round(MEASURED + IMPUTED, 4)}
assert SPEND['total_usd'] <= AUTHORIZED_USD, (SPEND['total_usd'], AUTHORIZED_USD)

print(f'{len(rows)} rows, {len(blank)} blank, {len(errors)} errored  {OUTPUT_SHA[:12]}')
print(f'${SPEND["total_usd"]:.4f} over {SPEND["total_calls"]} calls (${MEASURED:.4f} measured, '
      f'${IMPUTED:.4f} imputed over {UNRECORDED}) against ${AUTHORIZED_USD:.2f} authorized '
      f'and ${PROJECTED_TOTAL:.2f} projected')


1322 rows, 0 blank, 0 errored  19e34d284b64
$38.4708 over 1322 calls ($37.8904 measured, $0.5804 imputed over 20) against $50.00 authorized and $38.60 projected


---
## 9 — Manifest

The fourteen recorded rows are carried through untouched; this row is appended and its spend
is kept separate from the 2026-08-26 authorization.

In [37]:
import platform

MANIFEST = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
assert MANIFEST['eval_file']['sha256'] == TEST_SHA
assert len(MANIFEST['rows']) == 14 and NAME not in MANIFEST['rows']

MANIFEST['rows'][NAME] = {
    'condition': CONDITION, 'config': DERIVED['full']['path'], 'seconds': ELAPSED,
    'finished': datetime.now(timezone.utc).isoformat(), 'output_sha256': OUTPUT_SHA,
    'model': GEN['model'], 'effort': GEN['effort'],
    'thinking': 'always on in this tier; effort low is the floor, not off',
}
MANIFEST['derived_configs'].update({f'{SRC_CFG}::{tag}': d for tag, d in DERIVED.items()})
MANIFEST['spend']['per_row'][NAME] = SPEND
MANIFEST['spend'][f'{NAME}_authorization'] = {
    'authorized_usd': AUTHORIZED_USD, 'projected_usd': round(PROJECTED_TOTAL, 4),
    'actual_usd': SPEND['total_usd'], 'measured_usd': SPEND['measured_usd'],
    'imputed_usd': SPEND['unrecorded']['imputed_usd'],
    'imputed_note': f'{UNRECORDED} of {len(SEGMENTS)} calls have no token record; their cost is '
                    f'the stage rate times {UNRECORDED}, not a measurement',
    'runbook': 'notebooks/test_fable5_colab.ipynb',
    'separate_from': 'the 2026-08-26 generation authorization of $15',
}
MANIFEST['versions']['anthropic'] = __import__('anthropic').__version__
MANIFEST['versions']['python'] = platform.python_version()

MANIFEST_PATH.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(json.dumps({'rows': len(MANIFEST['rows']), NAME: MANIFEST['rows'][NAME],
                  'spend': MANIFEST['spend'][f'{NAME}_authorization']}, indent=2))


{
  "rows": 15,
  "fable5_sparse_knn": {
    "condition": "sparse_knn",
    "config": "configs/test/commercial_fable5_sparse_knn.full.yaml",
    "seconds": 5719.9,
    "finished": "2026-09-04T19:37:02.288539+00:00",
    "output_sha256": "19e34d284b64292a6d422bc924f7040b18370c271f169c7b667fd9cfa0ed2c4b",
    "model": "claude-fable-5-1",
    "effort": "low",
    "thinking": "always on in this tier; effort low is the floor, not off"
  },
  "spend": {
    "authorized_usd": 50,
    "projected_usd": 38.6029,
    "actual_usd": 38.4708,
    "measured_usd": 37.8904,
    "imputed_usd": 0.5804,
    "imputed_note": "20 of 1322 calls have no token record; their cost is the stage rate times 20, not a measurement",
    "runbook": "notebooks/test_fable5_colab.ipynb",
    "separate_from": "the 2026-08-26 generation authorization of $15"
  }
}


---
## 10 — Seal

In [38]:
# The fourteen prior rows still hash to what the manifest recorded for them.
for name, row in MANIFEST['rows'].items():
    if name == NAME:
        continue
    path = Path(f'outputs/{name}_{SPLIT}.jsonl')
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row['output_sha256'], name
assert len(MANIFEST['rows']) == 15, len(MANIFEST['rows'])

assert hashlib.sha256(EVAL_FILE.read_bytes()).hexdigest() == TEST_SHA, 'test.jsonl changed'
assert hashlib.sha256(RARITY.read_bytes()).hexdigest() == RARITY_SHA, 'the rarity list changed'
for f, digest in MANIFEST['index']['sha256'].items():
    assert hashlib.sha256((INDEX / f).read_bytes()).hexdigest() == digest, f

dirty = subprocess.run(['git', 'status', '--porcelain', 'configs', 'outputs', 'results', 'data'],
                       capture_output=True, text=True).stdout.splitlines()
unexpected = [x for x in dirty if 'test' not in x]
assert not unexpected, unexpected
print(f'15 rows on {SPLIT}, ${SPEND["total_usd"]:.4f} spent on this one, no prior row touched')

AssertionError: ['?? configs/commercial_fable5_sparse_knn.yaml']

In [39]:
!tar -czf test_fable5_generation.tar.gz \
    outputs/fable5_sparse_knn_test.jsonl outputs/fable5_sparse_knn_test_usage.json \
    outputs/fable5_sparse_knn_test_stage_usage.json outputs/test_manifest.json \
    configs/test/commercial_fable5_sparse_knn.stage.yaml \
    configs/test/commercial_fable5_sparse_knn.full.yaml
!ls -la test_fable5_generation.tar.gz
!git status --short outputs results configs


-rw-r--r-- 1 prnamhr prnamhr 196893 Sep  4 22:41 test_fable5_generation.tar.gz
 M outputs/test_manifest.json
?? configs/commercial_fable5_sparse_knn.yaml
?? configs/test/commercial_fable5_sparse_knn.full.yaml
?? configs/test/commercial_fable5_sparse_knn.pilot.yaml
?? configs/test/commercial_fable5_sparse_knn.stage.yaml
?? outputs/fable5_sparse_knn_test.jsonl
?? outputs/fable5_sparse_knn_test_stage_usage.json
?? outputs/fable5_sparse_knn_test_usage.json
